# 201 · Schema evolution lab

This notebook goes with the article
[Schema evolution](https://leo-gan.github.io/GLD.SerializerBenchmark/theory/201/schema-evolution/).

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/leo-gan/GLD.SerializerBenchmark/blob/master/docs/theory/notebooks/201/schema_evolution.ipynb)

In production, old and new versions of a service run at the same time during a rollout.
Writers and readers are often **not** upgraded in lockstep. Safe change means planning for both directions:
a new writer talking to an old reader, and an old writer talking to a new reader.

You will practice a small additive change: version 2 adds an optional `email` field as field number `3`.
**Never reuse a field number for a new meaning later.**

> **A note on numbers:** sizes and timings in these notebooks are only illustrations. For measured library comparisons on this project’s harness, use the suite [Results](https://leo-gan.github.io/GLD.SerializerBenchmark/) pages.


In [ ]:
from __future__ import annotations
from dataclasses import dataclass, field
from typing import Optional, Tuple


class WireError(Exception):
    pass


def encode_varint(u: int) -> bytes:
    out = bytearray()
    while u > 0x7F:
        out.append((u & 0x7F) | 0x80)
        u >>= 7
    out.append(u & 0x7F)
    return bytes(out)


def decode_varint(buf: bytes, i: int = 0) -> Tuple[int, int]:
    value = shift = nread = 0
    while True:
        if i >= len(buf):
            raise WireError("truncated")
        b = buf[i]
        i += 1
        nread += 1
        if nread > 10:
            raise WireError("overlong")
        value |= (b & 0x7F) << shift
        if b < 0x80:
            return value, i
        shift += 7


def encode_key(fn: int, wt: int) -> bytes:
    return encode_varint((fn << 3) | wt)


def require(buf: bytes, i: int, n: int):
    if i + n > len(buf):
        raise WireError("truncated payload")
    return buf[i : i + n], i + n



## Version 1 and version 2 of the message

- **Version 1** understands `id` (field 1) and `name` (field 2).
- **Version 2** keeps those fields and **adds** optional `email` as field 3.

Encoders omit empty default values in the style of Protocol Buffers proto3.
Decoders should accept fields in any order.


In [ ]:
@dataclass
class UserV1:
    id: int = 0
    name: str = ""


@dataclass
class UserV2:
    id: int = 0
    name: str = ""
    email: str = ""  # new optional field


def encode_v1(u: UserV1) -> bytes:
    out = bytearray()
    if u.id:
        out += encode_key(1, 0) + encode_varint(u.id)
    if u.name:
        b = u.name.encode()
        out += encode_key(2, 2) + encode_varint(len(b)) + b
    return bytes(out)


def encode_v2(u: UserV2) -> bytes:
    out = bytearray(encode_v1(UserV1(u.id, u.name)))
    if u.email:
        b = u.email.encode()
        out += encode_key(3, 2) + encode_varint(len(b)) + b
    return bytes(out)


def decode_v1(buf: bytes) -> UserV1:
    i = 0
    u = UserV1()
    while i < len(buf):
        key, i = decode_varint(buf, i)
        fn, wt = key >> 3, key & 7
        if wt == 0:
            v, i = decode_varint(buf, i)
            if fn == 1:
                u.id = v
        elif wt == 2:
            n, i = decode_varint(buf, i)
            payload, i = require(buf, i, n)
            if fn == 2:
                u.name = payload.decode()
            # fn == 3 or other: skip (old reader + new writer)
        elif wt == 1:
            _, i = require(buf, i, 8)
        elif wt == 5:
            _, i = require(buf, i, 4)
        else:
            raise WireError("bad wt")
    return u


def decode_v2(buf: bytes) -> UserV2:
    i = 0
    u = UserV2()
    while i < len(buf):
        key, i = decode_varint(buf, i)
        fn, wt = key >> 3, key & 7
        if wt == 0:
            v, i = decode_varint(buf, i)
            if fn == 1:
                u.id = v
        elif wt == 2:
            n, i = decode_varint(buf, i)
            payload, i = require(buf, i, n)
            if fn == 2:
                u.name = payload.decode()
            elif fn == 3:
                u.email = payload.decode()
        elif wt == 1:
            _, i = require(buf, i, 8)
        elif wt == 5:
            _, i = require(buf, i, 4)
        else:
            raise WireError("bad wt")
    return u



## Two rollout scenarios

**New writer, old reader:** version 2 encodes `email`, but the version 1 decoder only knows fields 1 and 2.
It should keep `id` and `name` and **skip** the unknown email field.

**Old writer, new reader:** version 1 never sends `email`.
The version 2 decoder should still succeed and leave `email` as an empty default.

Both directions matter whenever you cannot guarantee that every reader upgrades first (or every writer upgrades first).


In [ ]:
# New writer → old reader: old reader must ignore email
v2_bytes = encode_v2(UserV2(id=1, name="Ada", email="ada@example.com"))
old_view = decode_v1(v2_bytes)
assert old_view == UserV1(id=1, name="Ada")
print("OK writer v2 → reader v1:", old_view)

# Old writer → new reader: email defaults empty
v1_bytes = encode_v1(UserV1(id=1, name="Ada"))
new_view = decode_v2(v1_bytes)
assert new_view == UserV2(id=1, name="Ada", email="")
print("OK writer v1 → reader v2:", new_view)

# Breaking pattern demo: reusing field 2 for email would corrupt name — DO NOT
print("Never repurpose field numbers or JSON property meanings in place.")



## A short JSON contrast

Many JSON services evolve by allowing extra properties: a simple version 1 reader can ignore `email`.
That looks similar to “skip unknown fields,” but JSON still has weaker discipline around types and stable identity.

Renaming or removing a property without a migration plan still breaks consumers.
Additive change with documented defaults is the safer default habit.


In [ ]:
import json

def read_v1_json(s: str) -> dict:
    d = json.loads(s)
    return {"id": d.get("id", 0), "name": d.get("name", "")}


payload_v2 = json.dumps({"id": 1, "name": "Ada", "email": "ada@example.com"})
print("v1 view of v2 JSON:", read_v1_json(payload_v2))



## Takeaways

Prefer optional fields that you **add**, and document defaults clearly.
Do not repurpose Protocol Buffers field numbers (or silent JSON renames) for a new meaning.
“Forward” and “backward” compatibility are directions: know which way your deploy process needs to work.

**Next:** [Dynamic vs IDL binary](./dynamic_vs_idl_binary.ipynb)
